In [5]:
"""
Build analytic dataset for replicating Howell, Whitehead, & Korver-Glenn (2023)
"Still Separate and Unequal" using the 2023 NYC Housing Vacancy Survey (NYCHVS)
in place of the American Housing Survey + ACS.

Inputs (NYCHVS 2023 PUF subsets):
    - all_units_dataset_subset.csv
    - occupied_dataset_subset.csv
    - person_dataset_subset.csv

Output:
    - howell_replication_nychvs.csv

Key build decisions (set in conversation with the user):
    1. Income eligibility: <=80% AMI for the NYC HMFA (2023 HUD limits) by HHSIZE,
       OR receiving any rental assistance (RENTASSIST==1).
    2. Subsidy categories: 3 groups (Public housing / Voucher / Other) plus
       Unsubsidized. RENTASSIST==-1 (not reported) coded as MISSING for SUBSIDY,
       except where CSR==5 (public housing), which is determined structurally.
    3. Unsafe conditions: NYCHVS HPROBCOUNT (range 0-7, max observed = 6),
       with -1 ('not computed because some component item missing') coded as
       missing. Not capped: raw skewness in our sample is 0.84 (mild), and a
       max of 6 doesn't have the long right tail Howell's cap-at-4 was
       correcting for in the AHS (which counted 15 conditions, range 0-15).
    4. Affordability denominator: HHINC_REC1 (annual). Households with income==0
       have AFFORDABILITY set to NaN.
    5. Geographic isolation: dropped (NYCHVS lacks census-tract geography).
    6. Contextual factors (poverty rate, vacancy rate, mean income) computed at
       the BORO level, weighted by FW, from the full NYCHVS sample (not the
       eligible subset).
    7. Citizenship control: dropped (not provided in the subset).
    8. Implausible-cost rule (Howell): drop households where rent*12 > 1.5x income.
    9. YEARBUILT and UNITS retained as ordinal NYCHVS codes (no midpoint imputation).
    10. RENTASSIST==-1 -> SUBSIDY missing; MARITALSTAT_P==-1 -> SINGLE missing.
    11. Replicate weights FW1..FW80 (housing-unit-level, used for SDR variance
        estimation per the NYCHVS docs) are carried through to the output if
        present in the input subsets. Borough-level contextual factors are
        computed using FW only, not per-replicate -- they are treated as fixed
        covariates downstream rather than estimated quantities with their own
        sampling variance.
"""

'\nBuild analytic dataset for replicating Howell, Whitehead, & Korver-Glenn (2023)\n"Still Separate and Unequal" using the 2023 NYC Housing Vacancy Survey (NYCHVS)\nin place of the American Housing Survey + ACS.\n\nInputs (NYCHVS 2023 PUF subsets):\n    - all_units_dataset_subset.csv\n    - occupied_dataset_subset.csv\n    - person_dataset_subset.csv\n\nOutput:\n    - howell_replication_nychvs.csv\n\nKey build decisions (set in conversation with the user):\n    1. Income eligibility: <=80% AMI for the NYC HMFA (2023 HUD limits) by HHSIZE,\n       OR receiving any rental assistance (RENTASSIST==1).\n    2. Subsidy categories: 3 groups (Public housing / Voucher / Other) plus\n       Unsubsidized. RENTASSIST==-1 (not reported) coded as MISSING for SUBSIDY,\n       except where CSR==5 (public housing), which is determined structurally.\n    3. Unsafe conditions: NYCHVS HPROBCOUNT (range 0-7, max observed = 6),\n       with -1 (\'not computed because some component item missing\') coded as\

In [6]:
import numpy as np
import pandas as pd

In [7]:
# ---------------------------------------------------------------------------
# 0. Paths
# ---------------------------------------------------------------------------
INPUT_DIR  = "../data/processed"
OUTPUT_CSV = "../data/processed/howell_replication_nychvs.csv"

# Housing-unit-level replicate weight column names (NYCHVS uses 80 replicates;
# see Sec. 5 of the 2023 NYCHVS Sample Design, Weighting, and Error Estimation).
N_REPLICATES    = 80
REP_WEIGHT_COLS = [f"FW{i}" for i in range(1, N_REPLICATES + 1)]

In [8]:
# ---------------------------------------------------------------------------
# 1. Load NYCHVS PUF subsets
# ---------------------------------------------------------------------------
all_units = pd.read_csv(f"{INPUT_DIR}/all_units_dataset_subset.csv", dtype={"CONTROL": str})
occupied  = pd.read_csv(f"{INPUT_DIR}/occupied_dataset_subset.csv",  dtype={"CONTROL": str})
person    = pd.read_csv(f"{INPUT_DIR}/person_dataset_subset.csv",    dtype={"CONTROL": str})

# Detect whether replicate weights are present in all_units (the housing-unit
# file is the authoritative source for FW and its 80 replicates). The unit-level
# replicates should match between all_units and occupied for the same CONTROL --
# we keep the all_units copy and drop occupied's during the merge below.
HAS_REPLICATES = all(c in all_units.columns for c in REP_WEIGHT_COLS)
if HAS_REPLICATES:
    print(f"Detected replicate weights {REP_WEIGHT_COLS[0]}..{REP_WEIGHT_COLS[-1]} "
          "in all_units. They will be carried through to the output CSV.")
else:
    missing = [c for c in REP_WEIGHT_COLS if c not in all_units.columns]
    print(f"Replicate weights not detected in all_units (missing {len(missing)} "
          "of 80). Output will contain only the main weight FW.")

Detected replicate weights FW1..FW80 in all_units. They will be carried through to the output CSV.


In [9]:
all_units

,CONTROL,CSR,OCC,YEARBUILT,UNITS,ROOMS,BORO,FW,FW1,FW2,...,FW71,FW72,FW73,FW74,FW75,FW76,FW77,FW78,FW79,FW80
0,12130002,2,1,5,10,8,2,434.673470,434.673470,433.101582,...,442.455446,127.631770,711.527866,132.998833,709.589086,130.988565,736.863209,129.455196,762.451636,119.404255
1,12130005,80,1,3,6,3,3,555.435361,555.435361,932.333825,...,169.052548,553.284761,942.616620,164.975190,931.214014,559.255649,554.254696,560.187235,161.939232,567.640705
2,12130008,80,1,7,10,4,3,238.775657,238.775657,231.828215,...,239.068885,234.129431,71.519095,399.007919,71.703011,234.722388,410.574605,70.158697,393.432862,246.590232
3,12130010,80,1,4,10,1,3,226.590865,226.590865,219.345692,...,230.859408,224.223475,224.530093,232.675383,379.403388,66.640274,222.069717,387.334049,225.783637,232.247668
4,12130011,2,1,2,8,3,3,526.067721,526.067721,152.723123,...,537.958596,521.546224,888.184005,542.202939,149.320506,530.269565,524.148922,518.288581,900.353246,532.630520
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9592,12158886,32,1,10,9,2,1,256.429010,256.429010,75.962910,...,249.642410,429.093077,264.682886,260.675178,255.321122,77.789638,253.021151,439.918701,259.420088,258.679813
9593,12158888,80,1,3,2,6,2,586.284150,586.284150,170.460078,...,177.331859,174.204259,167.746866,587.990824,592.679480,173.258906,170.266419,575.221657,171.783096,172.634492
9594,12158891,6,1,10,2,5,5,576.899607,576.899607,982.037847,...,1016.971770,568.040423,582.367243,585.770664,594.786202,580.336959,165.784492,983.903069,172.660540,571.598400
9595,12158893,80,1,8,10,4,3,242.169172,242.169172,70.692910,...,415.471520,71.921377,239.770131,244.641758,401.938011,243.534240,239.737898,71.827585,240.138474,411.038052


In [10]:
all_units.columns

Index(['CONTROL', 'CSR', 'OCC', 'YEARBUILT', 'UNITS', 'ROOMS', 'BORO', 'FW',
       'FW1', 'FW2', 'FW3', 'FW4', 'FW5', 'FW6', 'FW7', 'FW8', 'FW9', 'FW10',
       'FW11', 'FW12', 'FW13', 'FW14', 'FW15', 'FW16', 'FW17', 'FW18', 'FW19',
       'FW20', 'FW21', 'FW22', 'FW23', 'FW24', 'FW25', 'FW26', 'FW27', 'FW28',
       'FW29', 'FW30', 'FW31', 'FW32', 'FW33', 'FW34', 'FW35', 'FW36', 'FW37',
       'FW38', 'FW39', 'FW40', 'FW41', 'FW42', 'FW43', 'FW44', 'FW45', 'FW46',
       'FW47', 'FW48', 'FW49', 'FW50', 'FW51', 'FW52', 'FW53', 'FW54', 'FW55',
       'FW56', 'FW57', 'FW58', 'FW59', 'FW60', 'FW61', 'FW62', 'FW63', 'FW64',
       'FW65', 'FW66', 'FW67', 'FW68', 'FW69', 'FW70', 'FW71', 'FW72', 'FW73',
       'FW74', 'FW75', 'FW76', 'FW77', 'FW78', 'FW79', 'FW80'],
      dtype='str')

In [11]:
occupied

,CONTROL,NOHEAT,NOHEAT_NUM,NOHOTWATER,LEAKS,MOLD,MUSTY,ELEVATOR_BROK,ELEVATOR_ALLBROK,RODENTS_UNIT,...,FW71,FW72,FW73,FW74,FW75,FW76,FW77,FW78,FW79,FW80
0,12130002,2,-2,1,2,2,5,1,1,2,...,442.455446,127.631770,711.527866,132.998833,709.589086,130.988565,736.863209,129.455196,762.451636,119.404255
1,12130005,2,-2,2,2,2,5,2,-2,2,...,169.052548,553.284761,942.616620,164.975190,931.214014,559.255649,554.254696,560.187235,161.939232,567.640705
2,12130008,2,-2,3,2,2,5,2,-2,2,...,239.068885,234.129431,71.519095,399.007919,71.703011,234.722388,410.574605,70.158697,393.432862,246.590232
3,12130010,2,-2,2,2,2,5,2,-2,2,...,230.859408,224.223475,224.530093,232.675383,379.403388,66.640274,222.069717,387.334049,225.783637,232.247668
4,12130011,2,-2,2,2,2,5,1,2,2,...,537.958596,521.546224,888.184005,542.202939,149.320506,530.269565,524.148922,518.288581,900.353246,532.630520
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8771,12158886,2,-2,2,2,2,5,1,2,1,...,249.642410,429.093077,264.682886,260.675178,255.321122,77.789638,253.021151,439.918701,259.420088,258.679813
8772,12158888,2,-2,2,2,2,5,-2,-2,2,...,177.331859,174.204259,167.746866,587.990824,592.679480,173.258906,170.266419,575.221657,171.783096,172.634492
8773,12158891,2,-2,2,2,2,5,-2,-2,2,...,1016.971770,568.040423,582.367243,585.770664,594.786202,580.336959,165.784492,983.903069,172.660540,571.598400
8774,12158893,-3,-3,-3,-3,-3,-3,-3,-3,1,...,415.471520,71.921377,239.770131,244.641758,401.938011,243.534240,239.737898,71.827585,240.138474,411.038052


In [12]:
occupied.columns[0:31]

Index(['CONTROL', 'NOHEAT', 'NOHEAT_NUM', 'NOHOTWATER', 'LEAKS', 'MOLD',
       'MUSTY', 'ELEVATOR_BROK', 'ELEVATOR_ALLBROK', 'RODENTS_UNIT',
       'RODENTS_BUILD', 'TOILET_BROK', 'ROACHES_NUM', 'WALLHOLES',
       'FLOORHOLES', 'PEELPAINT', 'PEELPAINT_LARGE', 'HPROBCOUNT',
       'RENT_AMOUNT', 'HHINC_REC1', 'RENTASSIST', 'RENTASSIST_VOUCHER',
       'HHSIZE', 'HH62PLUS', 'HHUNDER18', 'FW', 'FW1', 'FW2', 'FW3', 'FW4',
       'FW5'],
      dtype='str')

In [13]:
person

,CONTROL,RACE_P,RACEETH_P,RELATION_P,MARITALSTAT_P,SPOUSE_RES_P,SPOUSE_P,LNSPOUSE_P,LNPARTNER_P,PW,...,PW71,PW72,PW73,PW74,PW75,PW76,PW77,PW78,PW79,PW80
0,12130002,1,1,3,-2,-2,-2,-2,-2,406.212770,...,383.321453,124.489828,663.322538,118.583212,706.027375,110.421141,688.758431,122.383513,691.487193,121.200499
1,12130002,1,1,0,1,1,1,2,-2,434.673470,...,442.455446,127.631770,711.527866,132.998833,709.589086,130.988565,736.863209,129.455196,762.451636,119.404255
2,12130002,1,1,3,-2,-2,-2,-2,-2,406.212770,...,383.321453,124.489828,663.322538,118.583212,706.027375,110.421141,688.758431,122.383513,691.487193,121.200499
3,12130002,1,1,1,-2,-2,1,1,-2,434.673470,...,442.455446,127.631770,711.527866,132.998833,709.589086,130.988565,736.863209,129.455196,762.451636,119.404255
4,12130005,1,1,1,-2,-2,1,1,-2,555.435361,...,169.052548,553.284761,942.616620,164.975190,931.214014,559.255649,554.254696,560.187235,161.939232,567.640705
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20453,12158891,1,1,3,-2,-2,3,-2,-2,569.598977,...,1039.448893,541.407265,562.638856,637.448013,502.108532,629.914632,156.312966,1001.488482,154.037228,535.392556
20454,12158891,1,1,1,-2,-2,1,1,-2,576.899607,...,1016.971770,568.040423,582.367243,585.770664,594.786202,580.336959,165.784492,983.903069,172.660540,571.598400
20455,12158893,1,1,10,-2,-2,3,-2,-2,223.345895,...,321.550158,72.122335,220.485963,221.941051,366.956168,225.211468,238.324811,58.472787,248.717889,360.002113
20456,12158893,1,1,0,5,-2,3,-2,-2,242.169172,...,415.471520,71.921377,239.770131,244.641758,401.938011,243.534240,239.737898,71.827585,240.138474,411.038052


In [14]:
person.columns

Index(['CONTROL', 'RACE_P', 'RACEETH_P', 'RELATION_P', 'MARITALSTAT_P',
       'SPOUSE_RES_P', 'SPOUSE_P', 'LNSPOUSE_P', 'LNPARTNER_P', 'PW', 'PW1',
       'PW2', 'PW3', 'PW4', 'PW5', 'PW6', 'PW7', 'PW8', 'PW9', 'PW10', 'PW11',
       'PW12', 'PW13', 'PW14', 'PW15', 'PW16', 'PW17', 'PW18', 'PW19', 'PW20',
       'PW21', 'PW22', 'PW23', 'PW24', 'PW25', 'PW26', 'PW27', 'PW28', 'PW29',
       'PW30', 'PW31', 'PW32', 'PW33', 'PW34', 'PW35', 'PW36', 'PW37', 'PW38',
       'PW39', 'PW40', 'PW41', 'PW42', 'PW43', 'PW44', 'PW45', 'PW46', 'PW47',
       'PW48', 'PW49', 'PW50', 'PW51', 'PW52', 'PW53', 'PW54', 'PW55', 'PW56',
       'PW57', 'PW58', 'PW59', 'PW60', 'PW61', 'PW62', 'PW63', 'PW64', 'PW65',
       'PW66', 'PW67', 'PW68', 'PW69', 'PW70', 'PW71', 'PW72', 'PW73', 'PW74',
       'PW75', 'PW76', 'PW77', 'PW78', 'PW79', 'PW80'],
      dtype='str')

In [15]:
# ---------------------------------------------------------------------------
# 2. Restrict to occupied renter housing units
#    CSR codes treated as renters: 5 public housing, 32 rent-stabilized,
#    80 market renter, 90 rent-controlled, 97 other regulated renter.
# ---------------------------------------------------------------------------
RENTER_CSR = [5, 32, 80, 90, 97]
renters_au = all_units[
    (all_units["OCC"] == 1) & (all_units["CSR"].isin(RENTER_CSR))
].copy()

# Merge in occupied-unit characteristics. Drop FW and any replicate weights
# from occupied so that all_units' versions carry through (they should match
# row-for-row on CONTROL, but keeping a single authoritative copy avoids
# suffix-collision messiness).
weight_cols_to_drop = ["FW"] + [c for c in REP_WEIGHT_COLS if c in occupied.columns]
df = renters_au.merge(occupied.drop(columns=weight_cols_to_drop),
                      on="CONTROL", how="inner")

In [16]:
# ---------------------------------------------------------------------------
# 3. Attach respondent-level characteristics (RELATION_P == 0)
# ---------------------------------------------------------------------------
resp_cols = ["CONTROL", "RACE_P", "RACEETH_P",
             "MARITALSTAT_P", "SPOUSE_P", "SPOUSE_RES_P"]
resp = person.loc[person["RELATION_P"] == 0, resp_cols].copy()
df = df.merge(resp, on="CONTROL", how="left")


In [17]:
# ---------------------------------------------------------------------------
# 4. Apply income-eligibility filter
#    2023 NYC HMFA "Low-Income" (80% AMI) limits, applies to all 5 NYC boroughs.
#    Source: HUD income limits, effective May 15, 2023.
#    For HH > 8: HUD adds 8% of 4-person limit per additional person.
# ---------------------------------------------------------------------------
AMI80_2023_NYC_HMFA = {
    1: 79200, 2: 90500, 3: 101800, 4: 113100,
    5: 122150, 6: 131200, 7: 140250, 8: 149300,
}

def income_limit_for_hhsize(n: int) -> float:
    """80% AMI limit for a household of size n in NYC HMFA, 2023."""
    if n <= 8:
        return AMI80_2023_NYC_HMFA[n]
    return AMI80_2023_NYC_HMFA[8] + (n - 8) * 0.08 * AMI80_2023_NYC_HMFA[4]

df["INC_LIMIT_80AMI"]  = df["HHSIZE"].clip(lower=1).apply(income_limit_for_hhsize)
df["INCOME_ELIGIBLE"]  = (df["HHINC_REC1"] <= df["INC_LIMIT_80AMI"]).astype(int)
df["ANY_RENTASSIST"]   = (df["RENTASSIST"] == 1).astype(int)
df = df[(df["INCOME_ELIGIBLE"] == 1) | (df["ANY_RENTASSIST"] == 1)].copy()


In [18]:
# ---------------------------------------------------------------------------
# 5. Drop implausible cost cases (Howell): rent*12 > 1.5 x annual income.
#    Households with HHINC_REC1 == 0 are kept here but receive AFFORDABILITY=NaN
#    later. Using +inf for the ratio means they pass this filter only if rent==0.
# ---------------------------------------------------------------------------
df["_aff_check"] = np.where(
    df["HHINC_REC1"] > 0,
    (df["RENT_AMOUNT"] * 12) / df["HHINC_REC1"],
    np.inf,
)
df = df[df["_aff_check"] <= 1.5]

In [19]:
# ---------------------------------------------------------------------------
# 6. Outcomes
# ---------------------------------------------------------------------------
# 6a. Unsafe conditions: HPROBCOUNT (uncapped); -1 ("not computed" because
#     some component item is missing) -> NaN. See docstring for rationale.
df["UNSAFE_COND"] = df["HPROBCOUNT"].where(df["HPROBCOUNT"] >= 0, np.nan)

# 6b. Total monthly housing cost. NOTE: NYCHVS subset only has rent (no utilities).
#     Howell used rent + utilities + insurance (tothcamt). This is a known gap.
df["HOUSING_COST"] = df["RENT_AMOUNT"]

# 6c. Affordability: annualized rent / annual household income.
#     Income==0 -> NaN.
df["AFFORDABILITY"] = (df["RENT_AMOUNT"] * 12) / df["HHINC_REC1"].replace(0, np.nan)


In [20]:
# ---------------------------------------------------------------------------
# 7. Race / ethnicity (RACEETH_P, NYCHVS's combined race+Hispanic variable)
#    Mirrors Howell's grouping where possible. NHPI and AI/AN are combined in
#    NYCHVS (cat 5) -- cannot separate AI/AN as Howell did.
# ---------------------------------------------------------------------------
RACE_LABELS = {
    1: "White",       # White alone, non-Hispanic
    2: "Black",       # Black alone, non-Hispanic
    3: "Hispanic",    # Hispanic of any race
    4: "Asian",       # Asian alone, non-Hispanic
    5: "AIAN_NHPI",   # NHPI / AI/AN alone, non-Hispanic (combined)
    6: "Other_2plus", # Other or 2+ races, non-Hispanic
}
df["RACE"] = df["RACEETH_P"].map(RACE_LABELS)

In [21]:
# ---------------------------------------------------------------------------
# 8. Subsidy program (2 subsidized categories + unsubsidized; -1 -> missing)
#    Public housing is NO LONGER a SUBSIDY level -- it is carried entirely by
#    REGSTATUS=="Public_NYCHA" (section 10b) so public-housing information is
#    never encoded in two variables within the same model. Public units are
#    assigned SUBSIDY=="Unsubsidized" (the reference level), contributing no
#    SUBSIDY signal; their public-housing effect is estimated through REGSTATUS.
#    - Voucher:       RENTASSIST_VOUCHER == 1, not public housing.
#    - Other_subsidy: RENTASSIST == 1 but no voucher and not public housing.
#    - Unsubsidized:  RENTASSIST == 2 and not public housing, OR public housing.
#    - Missing:       Non-public-housing units with RENTASSIST == -1.
# ---------------------------------------------------------------------------
df["IS_VOUCHER"] = ((df["RENTASSIST_VOUCHER"] == 1) & (df["CSR"] != 5)).astype(int)
df["IS_OTHER"]   = (
    (df["RENTASSIST"] == 1)
    & (df["RENTASSIST_VOUCHER"] != 1)
    & (df["CSR"] != 5)
).astype(int)
# IS_UNSUB now also flags public-housing units (they take the Unsubsidized ref
# level in SUBSIDY; REGSTATUS carries their public-housing status).
df["IS_UNSUB"]   = (
    (df["CSR"] == 5)
    | ((df["CSR"] != 5) & (df["RENTASSIST"] == 2))
).astype(int)

def label_subsidy(row) -> object:
    if row["CSR"] == 5:
        return "Unsubsidized"   # public housing carried by REGSTATUS, not SUBSIDY
    if row["RENTASSIST"] == -1:
        return np.nan  # not reported
    if row["RENTASSIST_VOUCHER"] == 1:
        return "Voucher"
    if row["RENTASSIST"] == 1:
        return "Other_subsidy"
    if row["RENTASSIST"] == 2:
        return "Unsubsidized"
    return np.nan

df["SUBSIDY"] = df.apply(label_subsidy, axis=1)


In [22]:
# ---------------------------------------------------------------------------
# 9. Renter qualifications
#    - Older adult: any HH member 62+ (HH62PLUS == 1)
#    - Children:    any HH member <18 (HHUNDER18 == 1)
#    - Single:      respondent not legally married AND no unmarried partner in HH.
#                   MARITALSTAT_P == -1 (not reported) -> NaN.
# ---------------------------------------------------------------------------
df["OLDER_ADULT"] = (df["HH62PLUS"]   == 1).astype(int)
df["CHILDREN"]    = (df["HHUNDER18"]  == 1).astype(int)
df["SINGLE"] = np.where(
    df["MARITALSTAT_P"] == -1,
    np.nan,
    ((df["MARITALSTAT_P"] != 1) & (df["SPOUSE_P"] != 2)).astype(int),
)


In [23]:
# ---------------------------------------------------------------------------
# 10. Property characteristics (kept as NYCHVS ordinal codes)
# ---------------------------------------------------------------------------
YEARBUILT_LABELS = {
    1: "1900 or earlier", 2: "1901-1919", 3: "1920-1929", 4: "1930-1946",
    5: "1947-1959",       6: "1960-1973", 7: "1974-1979", 8: "1980-1989",
    9: "1990-1999",       10: "2000-2009", 11: "2010 or later",
}
UNITS_LABELS = {
    1: "1 unit",      2: "2 units",      3: "3 units",     4: "4-5 units",
    5: "6-9 units",   6: "10-12 units",  7: "13-19 units", 8: "20-49 units",
    9: "50-99 units", 10: "100+ units",
}

df["YEARBUILT_ORD"]   = df["YEARBUILT"].astype(int)
df["YEARBUILT_LABEL"] = df["YEARBUILT_ORD"].map(YEARBUILT_LABELS)
df["UNITS_BIN"]       = df["UNITS"].astype(int)
df["UNITS_LABEL"]     = df["UNITS_BIN"].map(UNITS_LABELS)
df["ROOMS_RAW"]       = df["ROOMS"]


In [24]:
# ---------------------------------------------------------------------------
# 10b. Rent-regulation status (from CSR)
#      Market rate is the reference category. Public housing (CSR 5) is given
#      its OWN level, "Public_NYCHA", rather than NaN. This is the single carrier
#      of public-housing information in the analysis: SUBSIDY (section 8) is
#      correspondingly stripped of its "Public" level (see note there), so public
#      housing is never encoded twice in the same model. Because public units
#      have a defined REGSTATUS level here, they are retained in every model that
#      uses REGSTATUS (including All renters) instead of being dropped by
#      complete-case filtering.
#        80 -> Market            (reference)
#        32 -> Rent_stabilized
#        90 -> Rent_controlled
#        97 -> Other_regulated
#         5 -> Public_NYCHA      (public housing; the sole public-housing carrier)
# ---------------------------------------------------------------------------
REGSTATUS_LABELS = {
    80: "Market",
    32: "Rent_stabilized",
    90: "Rent_controlled",
    97: "Other_regulated",
    5:  "Public_NYCHA",
}
df["REGSTATUS"] = df["CSR"].map(REGSTATUS_LABELS)

# Convenience indicator columns (mirrors the IS_* subsidy style).
df["IS_MARKET"]     = (df["CSR"] == 80).astype(int)
df["IS_STABILIZED"] = (df["CSR"] == 32).astype(int)
df["IS_CONTROLLED"] = (df["CSR"] == 90).astype(int)
df["IS_OTHERREG"]   = (df["CSR"] == 97).astype(int)
df["IS_PUBLIC_REG"] = (df["CSR"] == 5).astype(int)

In [25]:
# ---------------------------------------------------------------------------
# 5b. Drop implausible cost cases (Howell): where >100 percent of their income and >$1,000 monthly per room
# ---------------------------------------------------------------------------
df = df[~((df['_aff_check'] > 1.0) & (df['RENT_AMOUNT'] / df['ROOMS'] > 1000))]
df.drop(columns=["_aff_check"], inplace=True)

In [26]:
# ---------------------------------------------------------------------------
# 11. Borough-level contextual factors (weighted, computed from full NYCHVS)
#     - BORO_VACANCY:     net rental vacancy rate per HUD definition
#                         = vacant-for-rent / (occupied + vacant-for-rent)
#     - BORO_MEAN_INCOME: weighted mean of HHINC_REC1 across all occupied units
#     - BORO_POVERTY:     proxy poverty rate, share with HHINC_REC1 below the
#                         2022 federal poverty guideline for their HHSIZE
# ---------------------------------------------------------------------------
FPL_2022 = {1:13590, 2:18310, 3:23030, 4:27750,
            5:32470, 6:37190, 7:41910, 8:46630}

def fpl(n: int) -> float:
    """2022 federal poverty guideline for household of size n (48 contiguous states)."""
    if n <= 8:
        return FPL_2022[n]
    return FPL_2022[8] + (n - 8) * 4720

# Vacancy rate (numerator: vacant-for-rent; denominator: occupied + vacant-for-rent)
vac = all_units[all_units["OCC"].isin([1, 2])].copy()
vac["IS_VAC_RENT"] = (vac["OCC"] == 2).astype(int)
boro_vacancy = vac.groupby("BORO").apply(
    lambda g: (g["IS_VAC_RENT"] * g["FW"]).sum() / g["FW"].sum()
)

# Mean income across all occupied units
all_occ = all_units[all_units["OCC"] == 1].merge(
    occupied[["CONTROL", "HHINC_REC1", "HHSIZE"]], on="CONTROL", how="inner"
)
boro_mean_income = all_occ.groupby("BORO").apply(
    lambda g: (g["HHINC_REC1"] * g["FW"]).sum() / g["FW"].sum()
)

# Poverty proxy
all_occ["FPL_LIMIT"] = all_occ["HHSIZE"].clip(lower=1).apply(fpl)
all_occ["BELOW_FPL"] = (all_occ["HHINC_REC1"] < all_occ["FPL_LIMIT"]).astype(int)
boro_poverty = all_occ.groupby("BORO").apply(
    lambda g: (g["BELOW_FPL"] * g["FW"]).sum() / g["FW"].sum()
)

df["BORO_VACANCY"]     = df["BORO"].map(boro_vacancy)
df["BORO_MEAN_INCOME"] = df["BORO"].map(boro_mean_income)
df["BORO_POVERTY"]     = df["BORO"].map(boro_poverty)

BORO_LABELS = {1: "Bronx", 2: "Brooklyn", 3: "Manhattan",
               4: "Queens", 5: "Staten Island"}
df["BORO_NAME"] = df["BORO"].map(BORO_LABELS)


In [27]:
# ---------------------------------------------------------------------------
# 12. Final column selection
# ---------------------------------------------------------------------------
out_cols = [
    # Identifiers and weight
    "CONTROL", "BORO", "BORO_NAME", "FW",
    # Outcomes
    "UNSAFE_COND", "HPROBCOUNT", "HOUSING_COST", "AFFORDABILITY",
    # Race / ethnicity
    "RACE", "RACEETH_P", "RACE_P",
    # Subsidy program (Public level removed; carried by REGSTATUS instead)
    "SUBSIDY", "IS_VOUCHER", "IS_OTHER", "IS_UNSUB",
    # Rent-regulation status (from CSR; Public_NYCHA carries public housing)
    "REGSTATUS", "IS_MARKET", "IS_STABILIZED", "IS_CONTROLLED", "IS_OTHERREG",
    "IS_PUBLIC_REG",
    # Renter qualifications
    "OLDER_ADULT", "SINGLE", "CHILDREN",
    # Property characteristics (ordinal)
    "YEARBUILT_ORD", "YEARBUILT_LABEL", "UNITS_BIN", "UNITS_LABEL", "ROOMS_RAW",
    # Borough-level context
    "BORO_VACANCY", "BORO_MEAN_INCOME", "BORO_POVERTY",
    # Source variables retained for audit
    "HHINC_REC1", "HHSIZE", "RENT_AMOUNT", "RENTASSIST", "RENTASSIST_VOUCHER",
    "CSR", "HH62PLUS", "HHUNDER18", "MARITALSTAT_P", "SPOUSE_P",
]
# Append replicate weights at the end (kept together for readability)
if HAS_REPLICATES:
    out_cols += REP_WEIGHT_COLS
out = df[out_cols].copy()

# %%
# ---------------------------------------------------------------------------
# 13. Write output
# ---------------------------------------------------------------------------
out.to_csv(OUTPUT_CSV, index=False)
n_rep_cols = sum(c in out.columns for c in REP_WEIGHT_COLS)
print(f"\nWrote {len(out):,} rows and {len(out.columns)} columns to {OUTPUT_CSV}")
print(f"Replicate weight columns included: {n_rep_cols} of {N_REPLICATES}")
print("\nMissingness in analytic variables:")
analytic_cols = [
    "UNSAFE_COND", "HOUSING_COST", "AFFORDABILITY", "RACE", "SUBSIDY",
    "REGSTATUS",  # public housing now has its own level, not NaN
    "OLDER_ADULT", "SINGLE", "CHILDREN",
    "YEARBUILT_ORD", "UNITS_BIN", "ROOMS_RAW",
    "BORO_VACANCY", "BORO_MEAN_INCOME", "BORO_POVERTY",
]
print(out[analytic_cols].isna().sum().to_string())




Wrote 3,540 rows and 122 columns to ../data/processed/howell_replication_nychvs.csv
Replicate weight columns included: 80 of 80

Missingness in analytic variables:
UNSAFE_COND         339
HOUSING_COST          0
AFFORDABILITY         0
RACE                  0
SUBSIDY              45
REGSTATUS             0
OLDER_ADULT           0
SINGLE               70
CHILDREN              0
YEARBUILT_ORD         0
UNITS_BIN             0
ROOMS_RAW             0
BORO_VACANCY          0
BORO_MEAN_INCOME      0
BORO_POVERTY          0


In [28]:
out

,CONTROL,BORO,BORO_NAME,FW,UNSAFE_COND,HPROBCOUNT,HOUSING_COST,AFFORDABILITY,RACE,RACEETH_P,...,FW71,FW72,FW73,FW74,FW75,FW76,FW77,FW78,FW79,FW80
5,12130024,1,Bronx,511.852389,4.0,4,1500,0.346154,Black,2,...,497.416452,153.556908,523.489076,855.251576,150.517228,507.272519,879.733325,518.620716,151.002352,505.168052
6,12130028,1,Bronx,192.451639,2.0,2,635,0.334211,Hispanic,3,...,195.966220,57.030080,344.486786,188.025248,339.893186,54.958619,334.921991,188.261566,338.560652,56.387851
10,12130042,3,Manhattan,238.775657,0.0,0,2000,0.369231,White,1,...,408.116115,234.129431,244.181463,233.733427,71.703011,234.722388,410.574605,239.536776,67.502407,420.955857
11,12130046,3,Manhattan,55.213745,NaN,-1,2180,0.545000,White,1,...,55.489259,16.397829,93.322018,16.336848,53.681760,94.787064,16.009374,95.448914,54.750753,54.897137
12,12130048,1,Bronx,438.931415,4.0,4,1659,0.642194,Black,2,...,128.763665,769.992513,126.067944,445.324193,775.441907,128.404253,749.790670,429.459743,448.439294,440.739071
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6648,12158864,1,Bronx,513.876803,0.0,0,1475,0.486264,Black,2,...,526.182817,518.946685,149.322438,877.337598,151.106344,872.565356,151.052990,878.961137,154.405709,844.911745
6649,12158867,3,Manhattan,556.681109,3.0,3,495,0.708831,Hispanic,3,...,557.828079,550.616452,970.623275,541.965045,159.146405,570.179622,558.594984,548.499239,951.595565,566.376352
6650,12158868,1,Bronx,203.962884,1.0,1,1320,1.353846,Hispanic,3,...,344.096893,205.324791,61.999954,340.752964,204.547403,202.541521,204.842768,206.772630,211.439667,58.464650
6652,12158874,2,Brooklyn,107.223401,2.0,2,2012,1.029156,Hispanic,3,...,30.193766,33.634172,31.130198,107.406515,105.163130,30.930241,31.987776,106.676658,31.672554,30.874348
